In [ ]:
from z3 import *
from pprint import pprint
from fractions import Fraction
def Abs(x):
    return If(x >= 0,x,-x)

def Array2D(Type):
    return ArraySort(IntSort(),IntSort(), Type)

Box = Datatype('Box')
Box.declare('border')
Box.declare('wall')
Box.declare('empty')
Box = Box.create()
W=Box.wall
E=Box.empty
B=Box.border
field_arg=[[]]
dim,game=[9,6],"2_4a6p7b8a6b7a5b5p4a9_4"
#dim,game=[6,3],"a3b4c4_5c3b5a"
game=sum([[int(x)] if (ord(x)<=ord('9'))  else [-1]*int(ord(x)-ord('a')+1) for x in game],[])
for s in game:
    if len(field_arg[-1])==dim[1]:
        field_arg.append([])
    field_arg[-1].append(s)
field_arg[-1]+=[0]*(dim[1]-len(field_arg[-1]))
for x in field_arg:
    print(x)

s = Solver()


Field2D = Array2D(Box)
field  = Const('field', Field2D)


fx,fy=Ints("fx fy")
s.add(ForAll([fx, fy], If(Not(And(0<=fx,fx<dim[0],0<=fy,fy<dim[1])), field[fx,fy] == W, Or(field[fx,fy]==B,field[fx,fy]==E))))


is_connected = RecFunction("is_connected",Field2D, IntSort(),IntSort(), IntSort(),IntSort(),BoolSort())
field_param = FreshConst(Field2D)
x = FreshConst(IntSort())
y = FreshConst(IntSort())
xe = FreshConst(IntSort())
ye = FreshConst(IntSort())
RecAddDefinition(is_connected,[field_param,x,y,xe,ye],If(And(x==xe,y==ye),
                                             True,
                                          If(field_param[x,y]!=field_param[xe,ye],
                                             False,
                                          If(is_connected(field_param,x,y+1,xe,ye),
                                             True,
                                          If(is_connected(field_param,x,y-1,xe,ye),
                                             True,
                                          If(is_connected(field_param,x+1,y,xe,ye),
                                             True,
                                          If(is_connected(field_param,x-1,y,xe,ye),
                                             True,
                                          False
)))))))
ex,ey=-1,-1
for x in range(dim[0]):
    for y in range(dim[1]):
        if field_arg[x][y] >0:
            ex=x
            ey=y
x1,y1,x2,y2,x3,y3,x4,y4=Ints("x1 y1 x2 y2 x3 y3 x4 y4")
#s.add(ForAll([x1,y1,x2,y2], Implies(And(
#     And(0<=x1,x1<dim[0],0<=y1,y1<dim[1]),
#     And(0<=x2,x2<dim[0],0<=y2,y2<dim[1]),
#     True==field[x2,y2],
#     True==field[x1,y1]),            is_connected(field,x1,y1,x2,y2) )))
tx,ty=Ints("tx ty")
for x in range(dim[0]):
    for y in range(dim[1]):
        p=field_arg[x][y]
        s.add(Implies(field[x,y]==E, is_connected(field,x,y,ex,ey) ))

        s.add(Implies(field[x,y]==B, And(field[x+1,y]!=B,field[x-1,y]!=B,field[x,y-1]!=B,field[x,y+1]!=B)))
        if p<0:
            continue

        s.add(field[x,y]==E)
        s.add(Exists([x1,y1,x2,y2,x3,y3,x4,y4],And(
        And(x1==x,y<y1),
        And(x2==x,y2<y),
        And(x3<x,y3==y),
        And(x<x4,y4==y),
        Or(B==field[x1,y1],W==field[x1,y1]),
        Or(B==field[x2,y2],W==field[x2,y2]),
        Or(B==field[x3,y3],W==field[x3,y3]),
        Or(B==field[x4,y4],W==field[x4,y4]),
        ForAll([tx,ty],Implies(And(tx==x,y<ty,ty<y1), field[tx,ty]==E)),
        ForAll([tx,ty],Implies(And(tx==x,y2<ty,ty<y), field[tx,ty]==E)),
        ForAll([tx,ty],Implies(And(ty==y,x3<tx,tx<x), field[tx,ty]==E)),
        ForAll([tx,ty],Implies(And(ty==y,x<tx,tx<x4), field[tx,ty]==E)),
        (y1-y-1)+(y-y2-1)+(x-x3-1)+(x4-x-1)+1==p
)))


solution=[
[B,E,E],
[E,E,E],
[E,B,E],
[E,E,E],
[B,E,B],
[E,E,E],
]
solution=[]
for x in range(len(solution)):
    for y in range(len(solution[x])):
        s.add(field[x,y]==solution[x][y])

# s.add(is_connected(field,0,0,0,1)==True)
# s.add(is_connected(field,0,1,0,2)==True)
# s.add(is_connected(field,0,0,0,3)==False)
# s.add(is_connected(field,0,0,0,2)==True)#TODO does not ends for second defination of is_connected

while True:
    st=s.check()
    if st==sat:
        m = s.model()

        solution=[[-1 for y in range(dim[1])] for x in range(dim[0])]
        for x in range(dim[0]):
            for y in range(dim[1]):
                solution[x][y]=m.eval(field[x,y])
        pprint(solution)
        #break
        s.add(Not(And([m.eval(field[x,y])==field[x,y]  for y in range(dim[1]) for x in range(dim[0]) ])))

    else:
        print(st)
        break